In [ ]:
import pandas as pd
import os


import pandas as pd
import json


from pathlib import Path
from glob import glob
from collections import defaultdict

_____________________
_____________________

# GET AND FORMAT DATA FOR BiGG

In [19]:
output_dir = '../../analyses_com/results/tables'

In [20]:
# ⚙️ Paramètres globaux
conditions = ["1st", "15min"]#, "1h"]
conditions_filename = ["com_bigg_1st_fluxes", "com_bigg_15min_fluxes"]#, "com_bigg_1h_fluxes"]
conditions_dict = {"com_bigg_1st_fluxes":"1st" , "com_bigg_15min_fluxes":"15min" }#, "com_bigg_1h_fluxes":"1h" }
params = ["equal"]
community_modes = ["global", "bisteps", "delsupset"]
time_limits = {"1st": 2 * 60 * 60, "15min": 15 * 60}#, "1h": 60 * 60}
communities = ["com_BiGG_2_2", "com_BiGG_2_3", "com_BiGG_2_4", "com_BiGG_4_3"]

In [21]:
# 🔍 Recherche des fichiers TSV
def find_tsv_files(base_dir):
    return list(Path(base_dir).rglob("*_fluxes_from_result.tsv"))

# 🏷️ Extraction des métadonnées depuis le chemin du fichier
def extract_metadata(file_path):
    parts = file_path.parts
    condition = next((conditions_dict[c] for c in conditions_filename if c in parts), None)
    param = next((p for p in params if p in parts), None)
    name = file_path.name
    community = next((c for c in communities if c in name), None)
    mode = next((m for m in community_modes if m in name), None)
    return community, mode, condition, param

# ⏱️ Formatage du temps lisible
def format_time(seconds, limit_seconds):
    if isinstance(seconds, str) and seconds == "Time out":
        return "Time out"
    if seconds > limit_seconds:
        return "Time out"
    h, m, s = int(seconds // 3600), int((seconds % 3600) // 60), int(seconds % 60)
    if h > 0:
        return f"{h}h {m}m {s}s"
    elif m > 0:
        return f"{m}m {s}s"
    else:
        return f"{s}s"
    
# 📂 Chargement des fichiers JSON par condition
def load_json_by_condition(json_dir):
    timers = {}
    for condition in conditions:
        path = Path(json_dir) / f"com_bigg_{condition}_data" / "data_solution_bigg_com.json"
        if path.exists():
            with open(path, "r") as f:
                timers[condition] = json.load(f)
    return timers

In [22]:
# 🧱 Fonction principale de construction du tableau final
def build_final_table(tsv_base_dir, json_base_dir):
    tsv_files = find_tsv_files(tsv_base_dir)
    timers_by_condition = load_json_by_condition(json_base_dir)
    results = []

    # Collect existing combinations from TSV files
    seen_combinations = set()

    for file in tsv_files:
        community, mode, condition, param = extract_metadata(file)
        if not all([community, mode, condition, param]):
            continue
        df = pd.read_csv(file, sep="\t")
        df.drop_duplicates(subset='model', keep='first', inplace=True)
        solution_ok = df[df["has_flux"] == True].shape[0]
        grounding_time=None
        try:
            timer_data = timers_by_condition[condition][community]
            total = timer_data["nb_solution"][mode]["reas"]
            solving_time = timer_data["timer"][mode]["reas"]["Solving time"]
            if 'Grounding time' in timer_data["timer"][mode]["reas"]:
                grounding_time = timer_data["timer"][mode]["reas"]["Grounding time"]
            else:
                grounding_time = 0
            if grounding_time and grounding_time!= "time out":
                solving_time += grounding_time
            time_str = format_time(solving_time, time_limits[condition])
        except Exception:
            total = None
            time_str = "NA"

        ratio_str = f"{solution_ok} / {total}" if total else "NA"

        results.append({
            "community": community,
            "condition": condition,
            "param": param,
            "mode": mode,
            "ratio": ratio_str,
            "time": time_str
        })

        # Track this combination as already handled
        seen_combinations.add((community, condition, param, mode))

    # Add missing combinations from timers
    for condition, comm_data in timers_by_condition.items():
        for community, data in comm_data.items():
            for param in ["equal"]:  # assumed possible values
                for mode in data["nb_solution"].keys():
                    combo = (community, condition, param, mode)
                    grounding_time=None
                    if combo not in seen_combinations:
                        try:
                            total = data["nb_solution"][mode]["reas"]
                            solving_time = data["timer"][mode]["reas"]["Solving time"]
                            if 'Grounding time' in timer_data["timer"][mode]["reas"]:
                                grounding_time = timer_data["timer"][mode]["reas"]["Grounding time"]
                            else:
                                grounding_time = 0
                            if grounding_time and grounding_time!= "time out":
                                solving_time += grounding_time
                            time_str = format_time(solving_time, time_limits[condition])
                        except Exception:
                            total = None
                            time_str = "NA"

                        ratio_str = f"NA / {total}" if total else "NA"

                        results.append({
                            "community": community,
                            "condition": condition,
                            "param": param,
                            "mode": mode,
                            "ratio": ratio_str,
                            "time": time_str
                        })

    records = []
    for community in communities:
        for condition in conditions:
            record = {"community": community, "condition": condition}
            for param in params:
                for mode in community_modes:
                    match = next((r for r in results if r["community"] == community and
                                  r["condition"] == condition and r["param"] == param and r["mode"] == mode), None)

                    if match:
                        record[(param, mode, "ratio")] = match["ratio"]
                        record[(param, mode, "time")] = match["time"]
                    else:
                        record[(param, mode, "ratio")] = "NA"
                        record[(param, mode, "time")] = "NA"
                    
            records.append(record)

    df_final = pd.DataFrame(records)
    base_cols = ["community", "condition"]
    multi_cols = pd.MultiIndex.from_product(
        [params, community_modes, ["ratio", "time"]],
        names=["param", "mode", "metric"]
    )
    df_final = df_final[base_cols + list(multi_cols)]
    df_final["condition"] = pd.Categorical(df_final["condition"], categories=conditions, ordered=True)
    df_final.sort_values(by=["community", "condition"], inplace=True)
    df_final.reset_index(drop=True, inplace=True)

    return df_final

In [23]:
# ▶️ Exécution : construire le tableau et exporter
tsv_directory = "../../analyses_com/results"
json_directory = "../../analyses_com/results"

tableau = build_final_table(tsv_directory, json_directory)
#tableau.to_excel("tableau_fluxes.xlsx", index=False)

# Display or save
tableau.to_csv(f"{output_dir}/com_bigg_reasoning_table.csv", sep="\t", index=True)
tableau


/tmp/ipykernel_65431/4198211612.py:14: DtypeWarning: Columns (13,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep="\t")
/tmp/ipykernel_65431/4198211612.py:14: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep="\t")


,community,condition,"(equal, global, ratio)","(equal, global, time)","(equal, bisteps, ratio)","(equal, bisteps, time)","(equal, delsupset, ratio)","(equal, delsupset, time)"
0,com_BiGG_2_2,1st,0 / 1,3m 3s,1 / 1,3s,1 / 1,1m 28s
1,com_BiGG_2_2,15min,35756 / 116082,Time out,289 / 18139,Time out,170 / 214,Time out
2,com_BiGG_2_3,1st,1 / 1,2m 2s,1 / 1,14s,1 / 1,1m 18s
3,com_BiGG_2_3,15min,119200 / 141733,Time out,18952 / 19385,Time out,219 / 229,Time out
4,com_BiGG_2_4,1st,1 / 1,2m 15s,0 / 1,5s,0 / 1,1m 22s
5,com_BiGG_2_4,15min,90892 / 123939,Time out,407 / 16807,Time out,0 / 165,Time out
6,com_BiGG_4_3,1st,0 / 1,12m 51s,0 / 1,16s,0 / 1,6m 53s
7,com_BiGG_4_3,15min,5168 / 57013,Time out,1436 / 2625,Time out,25 / 140,Time out


______________________________
______________________________

# IDENTIFY IF SOLUTION OF DELETE SUPERSET IS A SUPERSET OF BISTEPS

In [24]:
# ⚙️ Paramètres globaux
conditions = ["1st", "15min"]#, "1h"]
conditions_filename = ["com_bigg_1st_fluxes", "com_bigg_15min_fluxes"]#, "com_bigg_1h_fluxes"]
conditions_dict = {"com_bigg_1st_fluxes":"1st" , "com_bigg_15min_fluxes":"15min"}# , "com_bigg_1h_fluxes":"1h"}
parameter = "equal"
community_modes = ["delsupset", "bisteps"]

In [25]:
# 🔍 Utilitaires
def extract_condition(path):
    for cond in conditions:
        if cond in path:
            return cond
    return "unknown"

def get_model_ok_from_tsv(tsv_file):
    df = pd.read_csv(tsv_file, sep='\t')
    return df[df["has_flux"] == True]["model"].unique().tolist(), len(df["model"].unique())

def extract_seeds_from_json(path):
    try:
        with open(path) as f:
            data = json.load(f)
        solutions = data["RESULTS"]["REASONING"]["SUBSET MINIMAL ENUMERATION"].get("solutions", {})
        seeds_dict = {}
        for model, content in solutions.items():
            try:
                idx = content.index("Set of seeds")
                seeds = set(content[idx + 1])
                seeds_dict[model] = seeds
            except ValueError:
                continue
        return seeds_dict
    except Exception:
        return {}


In [26]:
# 📂 Fichiers TSV : récupération des modèles model_ok
tsv_files = glob("../../analyses_com/results/*_fluxes/equal/*delsupset*from_result.tsv")
model_ok_dict = defaultdict(list)
model_total_dict = dict()

for file in tsv_files:
    basename = os.path.basename(file)
    community = basename.split("_rm_")[0]
    condition = extract_condition(file)
    if condition == "unknown":
        continue
    models, total = get_model_ok_from_tsv(file)
    model_ok_dict[(community, condition)] = models
    model_total_dict[(community, condition)] = total


In [27]:
# 📂 Fichiers JSON
json_delsupset_files = glob("../../analyses_com/results/*_results/*delsupset*reas*.json")
json_bisteps_files = glob("../../analyses_com/results/*_results/*bisteps*reas*.json")


# Associer fichiers à communauté + condition
def build_json_index(json_files):
    index = {}
    for path in json_files:
        community = os.path.basename(path).split("_rm_")[0]
        condition = extract_condition(path)
        index[(community, condition)] = path
    return index

delsupset_paths = build_json_index(json_delsupset_files)
bisteps_paths = build_json_index(json_bisteps_files)


In [ ]:
# Création du tableau final avec toutes les conditions
communities = sorted(set(k[0] for k in model_ok_dict.keys()) |
                     set(k[0] for k in delsupset_paths.keys()) |
                     set(k[0] for k in bisteps_paths.keys()))

summary_rows = []


for community in communities:
    for condition in conditions:
        model_list = model_ok_dict.get((community, condition), [])
        del_path = delsupset_paths.get((community, condition))
        bis_path = bisteps_paths.get((community, condition))

        if del_path and bis_path:
            seeds_delsup = extract_seeds_from_json(del_path)
            seeds_bisteps = extract_seeds_from_json(bis_path).values()
            n_ok = 0
            n_superset = 0
            for model in model_list:
                n_ok += 1
                # Check if the model is a superset of any bistep
                if any(bis.issubset(seeds_delsup[model]) for bis in seeds_bisteps):
                    n_superset += 1

            summary_rows.append({
                "community": community,
                "condition": condition,
                "n_model_ok": f"{n_ok} / {model_total_dict.get((community, condition))}",
                "n_superset_of_bisteps": n_superset
            })
        else:
            summary_rows.append({
                "community": community,
                "condition": condition,
                "n_model_ok": "NA",
                "n_superset_of_bisteps": "NA"
            })


In [29]:
# Construction du DataFrame final
df_summary_all = pd.DataFrame(summary_rows)
df_summary_all["condition"] = pd.Categorical(df_summary_all["condition"], categories=conditions, ordered=True)
df_summary_all.sort_values(by=["community", "condition"], inplace=True)
df_summary_all.reset_index(drop=True, inplace=True)
df_summary_all


,community,condition,n_model_ok,n_superset_of_bisteps
0,com_BiGG_2_2,1st,1 / 1,0
1,com_BiGG_2_2,15min,170 / 214,0
2,com_BiGG_2_3,1st,1 / 1,0
3,com_BiGG_2_3,15min,219 / 229,0
4,com_BiGG_2_4,1st,0 / 1,0
5,com_BiGG_2_4,15min,0 / 165,0
6,com_BiGG_4_3,1st,0 / 1,0
7,com_BiGG_4_3,15min,25 / 140,0
